# 📱 XGBoost Aplicado: Predicción de Precio de Smartphones

**Curso:** Machine Learning Aplicado 2026  
**Material teórico:** *XGBoost – Extreme Gradient Boosting: fundamentos, matemática y aplicaciones*  
**Autora del material:** Doctoral Student Gladys Choque Ulloa  
**Dataset:** Mobiles Dataset 2025 (Kaggle)  
**Algoritmo:** XGBoost (Extreme Gradient Boosting)

---

## 🎯 Objetivo del notebook

Aplicar el algoritmo XGBoost sobre un dataset real de smartphones para predecir su precio en USD, poniendo en práctica los conceptos matemáticos del material de clase:

- Función objetivo con regularización (Sección 3.1 del material)
- Aproximación de Taylor de segundo orden (Sección 3.2)
- Criterio de división de nodos (Sección 3.6)
- Hiperparámetros principales (Sección 4)

---

## 🗂️ Estructura del notebook

```
1. Contexto del problema
2. Carga y exploración de datos (EDA)
3. Preprocesamiento y Feature Engineering
4. Conexión con la matemática de XGBoost
5. Entrenamiento del modelo
6. Evaluación y métricas
7. Visualizaciones explicativas
8. Tuning de hiperparámetros
9. Conclusiones
```

---
## 📦 0. Instalación y librerías

Ejecuta esta celda una sola vez si no tienes xgboost instalado:

In [ ]:
# Instalar xgboost si no está disponible
# !pip install xgboost

# ─── Librerías estándar ───────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')

# ─── Visualización ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Estilo visual limpio para el aula
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

# ─── Machine Learning ────────────────────────────────────────────────────────
import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

print('✅ Todas las librerías cargadas correctamente')
print(f'   XGBoost versión: {xgb.__version__}')

---
## 1. 🌍 Contexto del problema

### ¿Qué queremos predecir?

El mercado de smartphones es uno de los más competitivos del mundo.  
Las empresas fijan precios basándose en especificaciones técnicas: RAM, cámara, batería, pantalla, etc.

**Pregunta de negocio:**  
> *¿Podemos predecir el precio de lanzamiento de un smartphone (en USD) a partir de sus especificaciones técnicas?*

### Tipo de problema

| Característica | Valor |
|---|---|
| Tipo | **Regresión supervisada** |
| Variable objetivo | Precio en USD (numérica continua) |
| Algoritmo | XGBoost Regressor |
| Dataset | 930 smartphones, 15 variables |

### Conexión con el material teórico

En la **Sección 5.2** del material (Ejemplo 2: Regresión), se explica cómo XGBoost usa pérdida cuadrática para regresión:

$$L = \frac{1}{2}(y_i - \hat{y}_i)^2$$

Con gradientes: $g_i = \hat{y}_i - y_i$ y hessiano: $h_i = 1$

Este es exactamente el escenario que vamos a trabajar.

---
## 2. 📊 Carga y Exploración de Datos (EDA)

In [ ]:
# ─── Carga del dataset ────────────────────────────────────────────────────────
# Encoding latin1 por caracteres especiales en el CSV original
df_raw = pd.read_csv('Mobiles_Dataset__2025_.csv', encoding='latin1')

print(f'📐 Dimensiones del dataset: {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas')
print(f'\n📋 Variables disponibles:')
for col in df_raw.columns:
    print(f'   • {col} ({df_raw[col].dtype})')

In [ ]:
# ─── Vista previa de los datos ────────────────────────────────────────────────
df_raw.head(8)

In [ ]:
# ─── Distribución por marca ───────────────────────────────────────────────────
brand_counts = df_raw['Company Name'].value_counts()

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(brand_counts.index, brand_counts.values,
              color=sns.color_palette('muted', len(brand_counts)))

# Etiquetas sobre cada barra
for bar, val in zip(bars, brand_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(val), ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title('📱 Cantidad de modelos por marca en el dataset', fontsize=14, fontweight='bold')
ax.set_xlabel('Marca')
ax.set_ylabel('Número de modelos')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f'\n✅ Total de marcas: {df_raw["Company Name"].nunique()}')

In [ ]:
# ─── Verificar valores nulos ───────────────────────────────────────────────────
print('🔍 Valores nulos por columna:')
nulls = df_raw.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else '   ✅ No hay valores nulos en el dataset')

---
## 3. 🔧 Preprocesamiento y Feature Engineering

Las variables están en formato de texto con unidades (ej: `"6GB"`, `"3,600mAh"`, `"USD 799"`).  
Necesitamos extraer los valores numéricos con expresiones regulares.

### Variables que usaremos

| Variable original | Variable procesada | Tipo |
|---|---|---|
| `RAM` | `ram_gb` | Numérica |
| `Battery Capacity` | `battery_mah` | Numérica |
| `Screen Size` | `screen_inches` | Numérica |
| `Mobile Weight` | `weight_g` | Numérica |
| `Front Camera` | `front_cam_mp` | Numérica |
| `Back Camera` | `back_cam_mp` | Numérica |
| `Launched Year` | `launched_year` | Numérica |
| `Company Name` | `brand_encoded` | Categórica → Numérica |
| **`Launched Price (USA)`** | **`price_usd`** | **Target** |


In [ ]:
# ─── Función auxiliar: extraer primer número de un string ─────────────────────
def extract_number(series: pd.Series) -> pd.Series:
    """
    Extrae el primer número (entero o decimal) de cada celda de texto.
    Elimina comas de separación de miles antes de parsear.
    Ejemplo: '3,600mAh' → 3600.0 | 'USD 799' → 799.0
    """
    return (
        series
        .astype(str)
        .str.replace(',', '', regex=False)       # quitar comas de miles
        .str.extract(r'([\d]+\.?[\d]*)')[0]      # primer número
        .astype(float)
    )

# ─── Construcción del dataframe limpio ────────────────────────────────────────
df = pd.DataFrame()

df['brand']          = df_raw['Company Name']
df['ram_gb']         = extract_number(df_raw['RAM'])
df['battery_mah']    = extract_number(df_raw['Battery Capacity'])
df['screen_inches']  = extract_number(df_raw['Screen Size'])
df['weight_g']       = extract_number(df_raw['Mobile Weight'])
df['front_cam_mp']   = extract_number(df_raw['Front Camera'])
df['back_cam_mp']    = extract_number(df_raw['Back Camera'])
df['launched_year']  = df_raw['Launched Year'].astype(float)
df['price_usd']      = extract_number(df_raw['Launched Price (USA)'])  # ← TARGET

# ─── Codificación de la variable categórica 'brand' ──────────────────────────
# Label Encoding: asigna un número entero a cada marca
le = LabelEncoder()
df['brand_encoded'] = le.fit_transform(df['brand'])

# Tabla de referencia: marca → código
brand_map = dict(zip(le.classes_, le.transform(le.classes_)))
print('🏷️  Codificación de marcas:')
for brand, code in brand_map.items():
    print(f'   {code:2d} → {brand}')

In [ ]:
# ─── Resumen estadístico post-procesamiento ───────────────────────────────────
print('📊 Estadísticas descriptivas del dataset procesado:\n')
df.drop(columns='brand').describe().round(2)

In [ ]:
# ─── Distribución del precio (variable objetivo) ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histograma
axes[0].hist(df['price_usd'], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df['price_usd'].mean(), color='tomato', linestyle='--', lw=2,
                label=f'Media: ${df["price_usd"].mean():.0f}')
axes[0].axvline(df['price_usd'].median(), color='gold', linestyle='--', lw=2,
                label=f'Mediana: ${df["price_usd"].median():.0f}')
axes[0].set_title('Distribución del precio (USD)', fontweight='bold')
axes[0].set_xlabel('Precio (USD)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# Boxplot por marca (top 8)
top_brands = df['brand'].value_counts().head(8).index
df_top = df[df['brand'].isin(top_brands)]
brand_order = df_top.groupby('brand')['price_usd'].median().sort_values(ascending=False).index

sns.boxplot(data=df_top, x='brand', y='price_usd', order=brand_order,
            palette='muted', ax=axes[1])
axes[1].set_title('Precio por marca (top 8)', fontweight='bold')
axes[1].set_xlabel('Marca')
axes[1].set_ylabel('Precio (USD)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# ─── Correlaciones con el precio ─────────────────────────────────────────────
features = ['ram_gb', 'battery_mah', 'screen_inches', 'weight_g',
            'front_cam_mp', 'back_cam_mp', 'launched_year', 'brand_encoded']

corr = df[features + ['price_usd']].corr()['price_usd'].drop('price_usd').sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['tomato' if v < 0 else 'steelblue' for v in corr.values]
bars = ax.barh(corr.index, corr.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)

for bar, val in zip(bars, corr.values):
    xpos = val + 0.01 if val >= 0 else val - 0.01
    ha = 'left' if val >= 0 else 'right'
    ax.text(xpos, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha=ha, fontsize=9)

ax.set_title('📈 Correlación de cada variable con el precio (USD)', fontweight='bold')
ax.set_xlabel('Correlación de Pearson')
ax.set_xlim(-0.5, 0.75)
plt.tight_layout()
plt.show()

---
## 4. 🧮 Conexión con la matemática de XGBoost

Antes de entrenar, repasemos los conceptos clave del material (Secciones 3.1 – 3.6).

### 4.1 Función objetivo (Ecuación 3 del material)

$$\mathcal{L}^{(t)} = \sum_{i=1}^{n} L\!\left(y_i,\; \hat{y}_i^{(t-1)} + f_t(x_i)\right) + \Omega(f_t)$$

Para nuestro problema de **regresión de precios**, la pérdida es cuadrática:

$$L = \frac{1}{2}(y_i - \hat{y}_i)^2$$

### 4.2 Gradiente y Hessiano (Ecuaciones 5 y 6)

Con pérdida cuadrática (Sección 5.2 del material):

$$g_i = \hat{y}_i - y_i \quad\quad h_i = 1$$

### 4.3 Regularización (Ecuación 7)

$$\Omega(f_t) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2$$

- **`gamma` (γ):** penaliza el número de hojas → árboles más simples
- **`reg_lambda` (λ):** regularización L2 sobre los pesos de las hojas

### 4.4 Peso óptimo de cada hoja (Ecuación 9)

$$w_j^* = -\frac{G_j}{H_j + \lambda}$$

### 4.5 Criterio de división (Ecuación 11)

$$\text{Gain} = \frac{1}{2}\left[\frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L+G_R)^2}{H_L+H_R+\lambda}\right] - \gamma$$

> 💡 **Regla:** si `Gain < 0`, no se realiza la división (poda automática).

In [ ]:
# ─── Demostración numérica: cálculo de Gain para un nodo ─────────────────────
# Supongamos que tenemos 6 smartphones en un nodo y evaluamos dividir por RAM ≤ 6GB

# Residuos simulados (precio_real - prediccion_actual)
# Grupo izquierdo: RAM ≤ 6GB  → 3 teléfonos baratos
# Grupo derecho:  RAM > 6GB   → 3 teléfonos premium
g_L = np.array([-200, -150, -180])  # gradientes izquierda
g_R = np.array([300, 400, 350])     # gradientes derecha

# Para pérdida cuadrática h_i = 1 siempre
h_L = np.ones(3)
h_R = np.ones(3)

# Parámetros de regularización
lam = 1.0   # lambda (reg_lambda)
gam = 0.1   # gamma

# Sumas de gradientes y hessianos por hoja (Sección 3.4 del material)
G_L, H_L = g_L.sum(), h_L.sum()
G_R, H_R = g_R.sum(), h_R.sum()
G_P, H_P = G_L + G_R, H_L + H_R  # padre (antes de dividir)

# Cálculo del Gain (Ecuación 11)
term_L = G_L**2 / (H_L + lam)
term_R = G_R**2 / (H_R + lam)
term_P = G_P**2 / (H_P + lam)
gain   = 0.5 * (term_L + term_R - term_P) - gam

# Pesos óptimos de cada hoja (Ecuación 9)
w_L = -G_L / (H_L + lam)
w_R = -G_R / (H_R + lam)

print('=' * 55)
print('  DEMOSTRACIÓN: Criterio de división de un nodo')
print('=' * 55)
print(f'  División: RAM ≤ 6GB (izquierda) | RAM > 6GB (derecha)')
print(f'  λ = {lam}  |  γ = {gam}')
print()
print(f'  G_L = {G_L:.1f}   H_L = {H_L:.1f}')
print(f'  G_R = {G_R:.1f}   H_R = {H_R:.1f}')
print()
print(f'  Término L  = G_L² / (H_L + λ) = {term_L:.2f}')
print(f'  Término R  = G_R² / (H_R + λ) = {term_R:.2f}')
print(f'  Término P  = G_P² / (H_P + λ) = {term_P:.2f}')
print()
print(f'  Gain = 0.5 × ({term_L:.2f} + {term_R:.2f} − {term_P:.2f}) − {gam}')
print(f'  Gain = {gain:.4f}  → {"✅ SE DIVIDE" if gain > 0 else "❌ NO SE DIVIDE"}')
print()
print(f'  Peso óptimo hoja izquierda  w*_L = {w_L:.2f}  (precio ajustado: ${w_L:.0f})')
print(f'  Peso óptimo hoja derecha    w*_R = {w_R:.2f}  (precio ajustado: +${w_R:.0f})')
print('=' * 55)

---
## 5. 🤖 Entrenamiento del modelo XGBoost

In [ ]:
# ─── Separación features / target ─────────────────────────────────────────────
FEATURES = ['ram_gb', 'battery_mah', 'screen_inches', 'weight_g',
            'front_cam_mp', 'back_cam_mp', 'launched_year', 'brand_encoded']
TARGET   = 'price_usd'

X = df[FEATURES]
y = df[TARGET]

print(f'✅ Features: {FEATURES}')
print(f'✅ Target:   {TARGET}')
print(f'✅ Shape X: {X.shape}  |  Shape y: {y.shape}')

In [ ]:
# ─── División entrenamiento / prueba (80% / 20%) ──────────────────────────────
# random_state=42 garantiza reproducibilidad (mismos resultados cada ejecución)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'📊 Conjunto de entrenamiento: {X_train.shape[0]} muestras')
print(f'📊 Conjunto de prueba:        {X_test.shape[0]} muestras')

In [ ]:
# ─── Definición del modelo XGBoost ────────────────────────────────────────────
#
# Referencia de hiperparámetros → Tabla 1 del material (Sección 4)
#
#   n_estimators  : número de árboles secuenciales (M en la Ecuación 1)
#   learning_rate : η — escala la contribución de cada árbol
#   max_depth     : profundidad máxima de cada árbol base
#   reg_lambda    : λ — regularización L2 sobre los pesos de las hojas (Ec. 7)
#   gamma         : γ — mínima ganancia para dividir un nodo (Ec. 11)
#   subsample     : fracción de muestras usadas por árbol (reduce varianza)
#   colsample_bytree: fracción de features usadas por árbol
#   random_state  : semilla para reproducibilidad

modelo_xgb = xgb.XGBRegressor(
    n_estimators      = 300,
    learning_rate     = 0.05,   # η pequeño → convergencia más suave
    max_depth         = 5,
    reg_lambda        = 1.5,    # λ: regularización L2
    gamma             = 0.2,    # γ: penalización por división
    subsample         = 0.8,    # 80% de muestras por árbol
    colsample_bytree  = 0.8,    # 80% de features por árbol
    random_state      = 42,
    verbosity         = 0       # sin logs durante entrenamiento
)

print('⚙️  Hiperparámetros configurados:')
params = modelo_xgb.get_params()
for key in ['n_estimators', 'learning_rate', 'max_depth', 'reg_lambda', 'gamma', 'subsample', 'colsample_bytree']:
    print(f'   {key:20s} = {params[key]}')

In [ ]:
# ─── Entrenamiento con monitoreo de la curva de aprendizaje ──────────────────
# eval_set permite ver cómo evolucionan el error de train y test en cada árbol
modelo_xgb.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False
)

print('✅ Modelo entrenado correctamente')
print(f'   Árboles construidos: {modelo_xgb.n_estimators}')

---
## 6. 📏 Evaluación y Métricas

In [ ]:
# ─── Predicciones ─────────────────────────────────────────────────────────────
y_pred_train = modelo_xgb.predict(X_train)
y_pred_test  = modelo_xgb.predict(X_test)

# ─── Métricas de evaluación ───────────────────────────────────────────────────
def evaluar(y_real, y_pred, nombre):
    mae  = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2   = r2_score(y_real, y_pred)
    mape = np.mean(np.abs((y_real - y_pred) / (y_real + 1e-9))) * 100
    print(f'  ── {nombre} ──')
    print(f'  MAE  (Error Absoluto Medio)  : ${mae:,.2f}')
    print(f'  RMSE (Raíz Error Cuadrático) : ${rmse:,.2f}')
    print(f'  MAPE (Error % Medio)         : {mape:.2f}%')
    print(f'  R²   (Coef. Determinación)   : {r2:.4f}')
    return mae, rmse, r2

print('\n📊 RESULTADOS DEL MODELO\n')
mae_tr, rmse_tr, r2_tr = evaluar(y_train, y_pred_train, '🔵 ENTRENAMIENTO')
print()
mae_te, rmse_te, r2_te = evaluar(y_test,  y_pred_test,  '🟢 PRUEBA')

print()
overfit = r2_tr - r2_te
print(f'  Diferencia R² (train - test): {overfit:.4f}  →  ', end='')
print('✅ Bien generalizado' if overfit < 0.1 else '⚠️  Posible sobreajuste')

---
## 7. 📈 Visualizaciones Explicativas

In [ ]:
# ─── Curva de aprendizaje (train vs test por árbol) ───────────────────────────
# Muestra cómo cada árbol nuevo reduce el error (concepto central de boosting)
results = modelo_xgb.evals_result()
epochs  = len(results['validation_0']['rmse'])
x_axis  = range(epochs)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(x_axis, results['validation_0']['rmse'], color='steelblue',
        alpha=0.7, label='Error Entrenamiento (RMSE)', lw=1.5)
ax.plot(x_axis, results['validation_1']['rmse'], color='tomato',
        alpha=0.9, label='Error Prueba (RMSE)', lw=2)

# Mínimo de test
best_iter = np.argmin(results['validation_1']['rmse'])
best_rmse = results['validation_1']['rmse'][best_iter]
ax.axvline(best_iter, color='gray', linestyle='--', lw=1)
ax.scatter(best_iter, best_rmse, color='tomato', s=80, zorder=5)
ax.annotate(f'Mínimo: ${best_rmse:,.0f}\n(árbol {best_iter})',
            xy=(best_iter, best_rmse),
            xytext=(best_iter + 20, best_rmse + 30),
            fontsize=9, arrowprops=dict(arrowstyle='->', color='gray'))

ax.set_title('📉 Curva de aprendizaje: reducción del error por árbol\n'
             '(Sección 5.2 del material — Ejemplo de Regresión)',
             fontweight='bold')
ax.set_xlabel('Número de árboles construidos')
ax.set_ylabel('RMSE (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

print(f'\n📌 Cada punto en la curva = un árbol nuevo que corrige los residuos del anterior')
print(f'   Esto es exactamente el proceso de boosting descrito en la Sección 2 del material.')

In [ ]:
# ─── Importancia de características ──────────────────────────────────────────
# Cuánto contribuyó cada variable a las decisiones de los árboles
importances = pd.Series(modelo_xgb.feature_importances_, index=FEATURES)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = sns.color_palette('Blues_r', len(importances))
bars = ax.barh(importances.index, importances.values, color=colors, edgecolor='white')

for bar, val in zip(bars, importances.values):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9, fontweight='bold')

ax.set_title('🔑 Importancia de características (Feature Importance)\n'
             '¿Qué variables influyen más en el precio?',
             fontweight='bold')
ax.set_xlabel('Importancia (F-Score normalizado)')
ax.set_xlim(0, importances.max() * 1.18)
plt.tight_layout()
plt.show()

print(f'\n📌 La variable más influyente es: "{importances.idxmax()}"')
print(f'   Esto significa que esa variable se usa más frecuentemente en las')
print(f'   divisiones de todos los árboles (criterio Gain, Ecuación 11 del material).')

In [ ]:
# ─── Predichos vs Reales ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

lim = (0, max(y_test.max(), y_pred_test.max()) * 1.05)

for ax, y_real, y_pred, title, color in [
    (axes[0], y_train, y_pred_train, '🔵 Entrenamiento', 'steelblue'),
    (axes[1], y_test,  y_pred_test,  '🟢 Prueba',        'seagreen')
]:
    ax.scatter(y_real, y_pred, alpha=0.5, color=color, s=25, edgecolors='white', lw=0.3)
    ax.plot(lim, lim, 'k--', lw=1.5, label='Predicción perfecta')
    r2 = r2_score(y_real, y_pred)
    ax.set_title(f'{title}   R² = {r2:.4f}', fontweight='bold')
    ax.set_xlabel('Precio real (USD)')
    ax.set_ylabel('Precio predicho (USD)')
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.legend(fontsize=8)

plt.suptitle('Precio Real vs. Precio Predicho por XGBoost', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Distribución de residuos ─────────────────────────────────────────────────
residuos = y_test.values - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histograma de residuos
axes[0].hist(residuos, bins=35, color='mediumpurple', edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='black', lw=1.5, linestyle='--')
axes[0].axvline(residuos.mean(), color='tomato', lw=2,
                label=f'Media: ${residuos.mean():.0f}')
axes[0].set_title('Distribución de Residuos (y_real − ŷ)', fontweight='bold')
axes[0].set_xlabel('Error (USD)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# Residuos vs predichos
axes[1].scatter(y_pred_test, residuos, alpha=0.5, color='mediumpurple',
                s=25, edgecolors='white', lw=0.3)
axes[1].axhline(0, color='black', lw=1.5, linestyle='--')
axes[1].set_title('Residuos vs. Valores Predichos', fontweight='bold')
axes[1].set_xlabel('Precio predicho (USD)')
axes[1].set_ylabel('Residuo (USD)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.suptitle('Análisis de Residuos del Modelo', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 8. 🎛️ Tuning de Hiperparámetros

Tal como menciona la **Sección 6.2 del material** (Limitaciones), XGBoost requiere **búsqueda sistemática de hiperparámetros**.

Usamos `GridSearchCV` con validación cruzada de 5 folds:

In [ ]:
# ─── Grid Search con validación cruzada ─────────────────────────────────────
# NOTA: Reducimos el grid para que sea rápido en clase.
# En producción se exploraría un espacio más amplio o se usaría Bayesian Search.

param_grid = {
    'n_estimators'  : [200, 300],
    'learning_rate' : [0.05, 0.10],
    'max_depth'     : [4, 5, 6],
    'gamma'         : [0.1, 0.3],
}

gs = GridSearchCV(
    estimator  = xgb.XGBRegressor(random_state=42, verbosity=0,
                                   subsample=0.8, colsample_bytree=0.8),
    param_grid = param_grid,
    cv         = 5,               # 5-fold cross validation
    scoring    = 'neg_root_mean_squared_error',
    n_jobs     = -1,              # usa todos los núcleos disponibles
    verbose    = 1
)

gs.fit(X_train, y_train)

print('\n🏆 Mejores hiperparámetros encontrados:')
for k, v in gs.best_params_.items():
    print(f'   {k:20s} = {v}')
print(f'\n   RMSE (CV) = ${-gs.best_score_:,.2f}')

In [ ]:
# ─── Modelo final con los mejores hiperparámetros ─────────────────────────────
modelo_final = gs.best_estimator_
y_pred_final = modelo_final.predict(X_test)

mae_f  = mean_absolute_error(y_test, y_pred_final)
rmse_f = np.sqrt(mean_squared_error(y_test, y_pred_final))
r2_f   = r2_score(y_test, y_pred_final)

print('\n📊 COMPARATIVA: Modelo base vs Modelo optimizado (sobre conjunto de prueba)')
print(f'{"Métrica":30s} {"Base":>15s} {"Optimizado":>15s} {"Mejora":>10s}')
print('-' * 72)
print(f'{"MAE  ($)":30s} {mae_te:>15,.2f} {mae_f:>15,.2f} {mae_te - mae_f:>+10,.2f}')
print(f'{"RMSE ($)":30s} {rmse_te:>15,.2f} {rmse_f:>15,.2f} {rmse_te - rmse_f:>+10,.2f}')
print(f'{"R²":30s} {r2_te:>15.4f} {r2_f:>15.4f} {r2_f - r2_te:>+10.4f}')

In [ ]:
# ─── Efecto de gamma sobre la complejidad (Ecuación 11) ────────────────────────
# Visualiza cómo γ controla el número de hojas y el error
gammas = [0, 0.1, 0.5, 1.0, 2.0, 5.0]
rmse_vals = []

for g in gammas:
    m = xgb.XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=5,
                          gamma=g, subsample=0.8, random_state=42, verbosity=0)
    m.fit(X_train, y_train)
    preds = m.predict(X_test)
    rmse_vals.append(np.sqrt(mean_squared_error(y_test, preds)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(gammas, rmse_vals, 'o-', color='darkorange', lw=2, markersize=8)
ax.fill_between(gammas, rmse_vals, min(rmse_vals), alpha=0.1, color='darkorange')

best_g = gammas[np.argmin(rmse_vals)]
best_r = min(rmse_vals)
ax.scatter(best_g, best_r, s=120, color='red', zorder=5,
           label=f'Mejor γ = {best_g}  (RMSE = ${best_r:,.0f})')

ax.set_title('Efecto del parámetro γ sobre el RMSE de prueba\n'
             '(γ controla la poda de nodos — Ecuación 11 del material)',
             fontweight='bold')
ax.set_xlabel('γ (gamma)')
ax.set_ylabel('RMSE (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

---
## 9. 🔮 Predicción de nuevos smartphones

In [ ]:
# ─── Predicción sobre ejemplos nuevos ─────────────────────────────────────────
# Aquí los alumnos pueden cambiar los valores y ver cómo varía la predicción

nuevos_phones = pd.DataFrame({
    'ram_gb'        : [4,    8,    12,   16],
    'battery_mah'   : [4000, 4500, 5000, 5000],
    'screen_inches' : [6.1,  6.5,  6.7,  6.8],
    'weight_g'      : [170,  190,  205,  220],
    'front_cam_mp'  : [8,    12,   32,   50],
    'back_cam_mp'   : [48,   108,  200,  200],
    'launched_year' : [2023, 2024, 2024, 2025],
    'brand_encoded' : [brand_map.get('Samsung', 0),
                       brand_map.get('Samsung', 0),
                       brand_map.get('Apple',   0),
                       brand_map.get('Apple',   0)]
})

predicciones = modelo_final.predict(nuevos_phones)

print('📱 Predicciones de precio para smartphones nuevos:\n')
etiquetas = ['Samsung (gama media)', 'Samsung (gama alta)',
             'Apple (gama alta)',    'Apple (flagship)']
for label, pred in zip(etiquetas, predicciones):
    print(f'   {label:25s} → Precio estimado: ${pred:,.2f} USD')

---
## 10. 📝 Conclusiones

### Lo que aprendimos en este notebook

| Concepto del material | Aplicación práctica |
|---|---|
| **Boosting secuencial** (Sección 2) | Cada árbol corrigió los residuos del anterior — lo vimos en la curva de aprendizaje |
| **Función objetivo con regularización** (Ec. 3) | Los parámetros `gamma` y `reg_lambda` evitaron sobreajuste |
| **Gradiente y Hessiano** (Ecs. 5 y 6) | Para regresión: $g_i = \hat{y}_i - y_i$ y $h_i = 1$ |
| **Criterio de división** (Ec. 11) | Calculamos manualmente el Gain de dividir por RAM |
| **Feature Importance** (Sección 6.1) | `brand_encoded` y `ram_gb` fueron las variables más influyentes |
| **Tuning de hiperparámetros** (Sección 6.2) | GridSearchCV mejoró el RMSE con validación cruzada |
| **Comparación con otros algoritmos** (Sección 7) | XGBoost superaría a un árbol CART por su regularización y aprendizaje secuencial |

### Reflexión final

> *"XGBoost no es magia: es matemática bien aplicada. Cada split que hace el modelo tiene detrás el cálculo del Gain. Cada hoja tiene el peso óptimo $w_j^* = -G_j/(H_j + \lambda)$. Cuando entiendes la ecuación, entiendes el modelo."*  
> — Adaptado del material de la Dra. Gladys Choque Ulloa

---

### 📚 Referencias

- Choque Ulloa, G. (2026). *XGBoost: Extreme Gradient Boosting — fundamentos, matemática y aplicaciones*. Material didáctico, Machine Learning Aplicado 2026.
- Chen, T., & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. *KDD 2016*.
- Friedman, J. H. (2001). Greedy function approximation: A gradient boosting machine. *The Annals of Statistics*.
- Dataset: Mobiles Dataset 2025, Kaggle.
- Documentación oficial: https://xgboost.readthedocs.io

---
## 🏋️ Ejercicios para los alumnos

1. **Gradiente manual:** Para un smartphone con precio real = $\$800$ y predicción actual = $\$650$, calcula $g_i$ y $h_i$ usando la pérdida cuadrática. ¿Qué indica el signo de $g_i$?

2. **Criterio de división:** En la celda de demostración de la Sección 4, cambia $\lambda = 5$. ¿Cómo afecta al Gain? ¿Por qué?

3. **Efecto del learning rate:** Cambia `learning_rate` a `0.01` y a `0.5` en el modelo base. ¿Qué observas en la curva de aprendizaje? ¿Cuántos árboles necesita cada configuración?

4. **Feature Engineering:** Crea una nueva feature `cam_total_mp = front_cam_mp + back_cam_mp`. ¿Mejora el R²? ¿Por qué podría ser útil?

5. **Predicción propia:** En la celda 9, diseña tu smartphone soñado con especificaciones reales y predice su precio. ¿Es cercano al precio real de ese modelo?

---
*Notebook elaborado para el curso Machine Learning Aplicado 2026 — basado en el material teórico de la Dra. Gladys Choque Ulloa*